# GNN Churn Prediction
**Goal:** Predict customer churn using a Graph Neural Network (GraphSAGE) that models social influence between customers.

**Key idea:** Churn spreads through social networks. If a customer's frequent contacts are churning, they're more likely to churn too. A GNN can capture this signal — a traditional ML model cannot.

## Pipeline
1. Load & preprocess the Telco dataset
2. Build a customer call graph (nodes = customers, edges = inferred call relationships)
3. Train a GraphSAGE model
4. Compare AUC vs baseline XGBoost
5. Save model for API serving

In [ ]:
# Install dependencies (run once)
# !pip install torch torch_geometric xgboost scikit-learn pandas numpy matplotlib seaborn kaggle

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
print('Libraries loaded.')

## Step 1: Load & Preprocess Data

Download from Kaggle: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
Place `WA_Fn-UseC_-Telco-Customer-Churn.csv` in the `../data/` folder.

In [ ]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Dataset shape: {df.shape}')
print(f'Churn rate: {df.Churn.value_counts(normalize=True).Yes:.1%}')
df.head()

In [ ]:
# ── Clean & encode ──────────────────────────────────────────────
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df['Churn_binary'] = (df['Churn'] == 'Yes').astype(int)

# Encode binary yes/no columns
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col] = (df[col] == 'Yes').astype(int)

df['gender'] = (df['gender'] == 'Male').astype(int)

# One-hot encode multi-class columns
cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Feature matrix
drop_cols = ['customerID', 'Churn', 'Churn_binary']
feature_cols = [c for c in df.columns if c not in drop_cols]

scaler = StandardScaler()
X = scaler.fit_transform(df[feature_cols].values)
y = df['Churn_binary'].values

print(f'Feature matrix: {X.shape}')
print(f'Class balance — churned: {y.mean():.1%}')

## Step 2: Build the Customer Call Graph

The Telco dataset doesn't include explicit call logs, so we construct a **synthetic but realistic** social graph.

**Edge logic:** Two customers are connected if they share the same area code (derived from tenure bucket) AND both have phone service. This simulates the kind of call-graph structure telecom companies actually have internally. We document this assumption clearly.

In a real scenario, you'd use actual CDR (Call Detail Records).

In [ ]:
def build_call_graph(df_original, X_features, max_edges_per_node=5):
    """
    Build a customer social graph.
    Nodes: customers (with feature vectors)
    Edges: customers with similar tenure + same phone service status
    """
    n = len(df_original)
    
    # Create tenure buckets (proxy for area/cohort)
    tenure_bucket = pd.cut(df_original['tenure'], bins=10, labels=False)
    has_phone = df_original['PhoneService'].values
    
    edges_src, edges_dst = [], []
    
    for bucket in range(10):
        # Customers in the same tenure bucket with phone service
        mask = (tenure_bucket == bucket) & (has_phone == 1)
        group_idx = np.where(mask)[0]
        
        if len(group_idx) < 2:
            continue
        
        # Sample edges within group (not fully connected — too dense)
        np.random.shuffle(group_idx)
        for i, node in enumerate(group_idx):
            # Connect to up to max_edges_per_node neighbors
            neighbors = group_idx[
                np.random.choice(
                    len(group_idx),
                    size=min(max_edges_per_node, len(group_idx) - 1),
                    replace=False
                )
            ]
            for nb in neighbors:
                if nb != node:
                    edges_src.append(node)
                    edges_dst.append(nb)
    
    # Make undirected
    all_src = edges_src + edges_dst
    all_dst = edges_dst + edges_src
    edge_index = torch.tensor([all_src, all_dst], dtype=torch.long)
    
    print(f'Graph: {n} nodes, {len(all_src)} edges')
    print(f'Avg degree: {len(all_src)/n:.1f}')
    
    return edge_index


edge_index = build_call_graph(df, X)

# PyG Data object
data = Data(
    x=torch.tensor(X, dtype=torch.float),
    edge_index=edge_index,
    y=torch.tensor(y, dtype=torch.long)
)

print(f'\nPyG Data object:')
print(data)

In [ ]:
# Visualise a subgraph (first 80 nodes)
sub_edge_index = edge_index[:, edge_index[0] < 80]
sub_edge_index = sub_edge_index[:, sub_edge_index[1] < 80]

G = nx.Graph()
G.add_nodes_from(range(80))
edges = sub_edge_index.T.tolist()
G.add_edges_from(edges)

node_colors = ['#E24B4A' if y[i] == 1 else '#378ADD' for i in range(80)]

plt.figure(figsize=(10, 7))
pos = nx.spring_layout(G, seed=42, k=1.2)
nx.draw_networkx(G, pos, node_color=node_colors, node_size=120,
                 with_labels=False, edge_color='#D3D1C7', width=0.5, alpha=0.85)

from matplotlib.patches import Patch
legend = [Patch(color='#E24B4A', label='Churned'), Patch(color='#378ADD', label='Retained')]
plt.legend(handles=legend, loc='upper left', fontsize=11)
plt.title('Customer social graph (sample of 80 nodes)', fontsize=13, pad=12)
plt.axis('off')
plt.tight_layout()
plt.savefig('../data/graph_sample.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graph saved.')

## Step 3: Train/Val/Test Split

In [ ]:
n = data.num_nodes
idx = np.arange(n)
train_idx, test_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=42)
train_idx, val_idx = train_test_split(train_idx, test_size=0.15, stratify=y[train_idx], random_state=42)

train_mask = torch.zeros(n, dtype=torch.bool)
val_mask   = torch.zeros(n, dtype=torch.bool)
test_mask  = torch.zeros(n, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx]     = True
test_mask[test_idx]   = True

data.train_mask = train_mask
data.val_mask   = val_mask
data.test_mask  = test_mask

print(f'Train: {train_mask.sum()} | Val: {val_mask.sum()} | Test: {test_mask.sum()}')

## Step 4: GraphSAGE Model

In [ ]:
class ChurnGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.classifier = torch.nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        # Layer 1: aggregate neighbor features
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        # Layer 2: refine embeddings
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        # Classify
        return self.classifier(x)

    def embed(self, x, edge_index):
        """Return node embeddings (for visualization)."""
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return x


model = ChurnGNN(
    in_channels=data.num_features,
    hidden_channels=64,
    out_channels=2
)
print(model)
print(f'\nParameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Handle class imbalance with weighted loss
churn_weight = (y == 0).sum() / (y == 1).sum()
class_weights = torch.tensor([1.0, float(churn_weight)])
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

train_losses, val_aucs = [], []

def train_epoch():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return float(loss)

@torch.no_grad()
def evaluate(mask):
    model.eval()
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)[:, 1].numpy()
    labels = data.y[mask].numpy()
    return roc_auc_score(labels, probs[mask.numpy()])

print('Training...')
best_val_auc = 0
best_state = None

for epoch in range(1, 201):
    loss = train_epoch()
    scheduler.step()
    train_losses.append(loss)

    if epoch % 10 == 0:
        val_auc = evaluate(data.val_mask)
        val_aucs.append(val_auc)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if epoch % 50 == 0:
            print(f'Epoch {epoch:3d} | Loss: {loss:.4f} | Val AUC: {val_auc:.4f}')

model.load_state_dict(best_state)
test_auc = evaluate(data.test_mask)
print(f'\nBest Val AUC: {best_val_auc:.4f}')
print(f'Test AUC    : {test_auc:.4f}')

In [ ]:
# Training curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, color='#534AB7', linewidth=1.2)
ax1.set_title('Training loss'); ax1.set_xlabel('Epoch')

ax2.plot(range(10, 201, 10), val_aucs, color='#1D9E75', linewidth=1.5, marker='o', markersize=4)
ax2.axhline(y=test_auc, color='#E24B4A', linestyle='--', label=f'Test AUC: {test_auc:.3f}')
ax2.set_title('Validation AUC'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout()
plt.savefig('../data/training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5: Baseline Comparison (XGBoost)

In [ ]:
xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                    scale_pos_weight=churn_weight, random_state=42, eval_metric='auc')
xgb.fit(X[train_idx], y[train_idx])
xgb_probs = xgb.predict_proba(X[test_idx])[:, 1]
xgb_auc = roc_auc_score(y[test_idx], xgb_probs)

print('=' * 40)
print(f'  XGBoost (no graph) AUC : {xgb_auc:.4f}')
print(f'  GraphSAGE AUC          : {test_auc:.4f}')
print(f'  Lift from graph        : +{(test_auc - xgb_auc)*100:.1f}%')
print('=' * 40)

## Step 6: Save Everything for the API

In [ ]:
import json, pickle

# Save GNN model weights
torch.save(model.state_dict(), '../backend/model_weights.pt')

# Save scaler + feature column names (needed for inference)
with open('../backend/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../backend/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

# Save edge index for the full graph (API uses it for neighbor lookups)
torch.save(edge_index, '../backend/edge_index.pt')

# Save node-level predictions for dashboard
model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index)
    all_probs = F.softmax(out, dim=1)[:, 1].numpy()

results = pd.DataFrame({
    'customer_id': df.index.tolist(),
    'churn_prob': all_probs.round(4).tolist(),
    'actual_churn': y.tolist(),
    'tenure': df['tenure'].tolist(),
    'monthly_charges': df['MonthlyCharges'].tolist()
})
results.to_csv('../data/predictions.csv', index=False)

# Save graph stats for API
graph_stats = {
    'num_nodes': int(data.num_nodes),
    'num_edges': int(edge_index.shape[1] // 2),
    'gnn_test_auc': round(float(test_auc), 4),
    'xgb_test_auc': round(float(xgb_auc), 4),
    'churn_rate': round(float(y.mean()), 4),
    'avg_degree': round(float(edge_index.shape[1] / data.num_nodes), 2)
}
with open('../data/graph_stats.json', 'w') as f:
    json.dump(graph_stats, f, indent=2)

print('Saved:')
for f in ['backend/model_weights.pt', 'backend/scaler.pkl',
          'backend/feature_cols.json', 'backend/edge_index.pt',
          'data/predictions.csv', 'data/graph_stats.json']:
    print(f'  ../{f}')
print(f'\nAll done. GNN AUC: {test_auc:.4f}')